# 🔌 LLM API 基础 —— 多 Provider 调用模式

本 Notebook 覆盖生产级 LLM API 调用的核心模式：从基础请求到流式输出、
Token 管理、错误重试、多 Provider 对比、成本追踪。

**学习目标：**
- 掌握 OpenAI Chat Completions API 的完整用法
- 实现流式输出（Streaming）与 token-by-token 处理
- 使用 tiktoken 精确计算和管控 Token 预算
- 实现生产级的指数退避 + 抖动重试策略
- 掌握 Anthropic Messages API 的等效模式
- 对比两大 Provider 的 API 设计差异
- 实现实时的成本追踪系统

**先决条件：** `pip install openai anthropic tiktoken`

---
## 1. OpenAI Chat Completions API —— 基本调用

### 核心参数
- `model` — 模型名称（gpt-4o, gpt-4o-mini 等）
- `messages` — 对话消息列表 [{role, content}, ...]
- `temperature` — 随机性（0.0-2.0，0 最确定性）
- `max_tokens` — 最大输出 token 数
- `top_p` — 核采样（nucleus sampling）
- `seed` — 确定性种子（用于可复现结果）
- `stop` — 停止序列

In [ ]:
"""
OpenAI Chat Completions API —— 基本调用
========================================

前置条件：设置环境变量 OPENAI_API_KEY
  export OPENAI_API_KEY="sk-..."

如果没有 API Key，本 cell 会自动跳过实际调用，使用模拟输出演示。
"""
import os
import json
from typing import List, Dict, Optional, Any

# 尝试导入 OpenAI SDK
try:
    from openai import OpenAI
    from openai.types.chat import ChatCompletion
    OPENAI_AVAILABLE = True
    print("✅ OpenAI SDK 已安装")
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI SDK 未安装。运行: pip install openai")
    print("   将使用模拟输出演示 API 用法。")

# ============================================================
# 检查 API Key
# ============================================================
api_key = os.getenv("OPENAI_API_KEY", "")
HAS_REAL_KEY = bool(api_key and api_key.startswith("sk-"))

if not HAS_REAL_KEY:
    print("ℹ️  未设置有效的 OPENAI_API_KEY，将使用模拟输出。")
    print("   export OPENAI_API_KEY='sk-your-key-here'")

print()

In [ ]:
# ============================================================
# 基本 Chat Completion 调用
# ============================================================

def basic_chat_completion(
    prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    max_tokens: int = 256,
    top_p: float = 1.0,
    seed: Optional[int] = 42,
    system_prompt: str = "你是一个有帮助的AI助手。请用中文回答。",
) -> Dict[str, Any]:
    """
    OpenAI Chat Completion 的基本调用封装。
    
    参数说明：
    - temperature=0.0: 最确定性的输出（代码生成、事实问答）
    - temperature=0.7-1.0: 创意性任务（写作、头脑风暴）
    - max_tokens: 控制输出长度，节省成本
    - seed: 配合 temperature=0 获得可复现输出
    - top_p: 0.1 更聚焦，0.9 更多样
    
    返回格式：
    {
        "content": "助手的回复文本",
        "model": "实际使用的模型",
        "usage": {"prompt_tokens": N, "completion_tokens": M, "total_tokens": T},
        "finish_reason": "stop" | "length" | "content_filter"
    }
    """
    if not HAS_REAL_KEY or not OPENAI_AVAILABLE:
        # 模拟输出
        return {
            "content": f"[模拟] 收到 prompt: '{prompt[:50]}...'，模型={model}, temp={temperature}",
            "model": model,
            "usage": {"prompt_tokens": 12, "completion_tokens": 20, "total_tokens": 32},
            "finish_reason": "stop",
        }
    
    client = OpenAI(api_key=api_key)
    
    response: ChatCompletion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        seed=seed,
    )
    
    return {
        "content": response.choices[0].message.content or "",
        "model": response.model,
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens if response.usage else 0,
            "completion_tokens": response.usage.completion_tokens if response.usage else 0,
            "total_tokens": response.usage.total_tokens if response.usage else 0,
        },
        "finish_reason": response.choices[0].finish_reason or "unknown",
    }


# 测试：不同 temperature 的效果对比
print("=" * 60)
print("Temperature 效果对比")
print("=" * 60)

prompt_creative = "用一句话描述春天。"

for temp, label in [(0.0, "确定性"), (0.5, "平衡"), (1.2, "创意性")]:
    result = basic_chat_completion(
        prompt=prompt_creative,
        model="gpt-4o-mini",
        temperature=temp,
        max_tokens=100,
    )
    print(f"\n🌡️  temperature={temp} ({label}):")
    print(f"   回复: {result['content'][:80]}...")
    print(f"   tokens: {result['usage']['total_tokens']}")

print()

# 测试：不同 max_tokens 控制输出长度
print("=" * 60)
print("max_tokens 截断效果")
print("=" * 60)

for mt in [50, 150, 300]:
    result = basic_chat_completion(
        prompt="请详细介绍人工智能的发展历史。",
        model="gpt-4o-mini",
        max_tokens=mt,
    )
    finish = result['finish_reason']
    print(f"  max_tokens={mt}: finish_reason='{finish}', "
          f"completion_tokens={result['usage']['completion_tokens']}")

print()
print("💡 finish_reason='length' 表示因 max_tokens 限制被截断")
print("💡 finish_reason='stop' 表示模型自然结束")

In [ ]:
# ============================================================
# 高级用法：Few-shot prompting、多轮对话
# ============================================================

def multi_turn_chat(
    messages: List[Dict[str, str]],
    model: str = "gpt-4o-mini",
) -> str:
    """
    多轮对话封装。
    
    messages 格式：
    [
        {"role": "system", "content": "你是一个..."},
        {"role": "user", "content": "你好"},
        {"role": "assistant", "content": "你好！有什么可以帮你的？"},
        {"role": "user", "content": "..."},
    ]
    
    在实际 RAG 项目中，messages 就是从对话历史构建的。
    """
    if not HAS_REAL_KEY or not OPENAI_AVAILABLE:
        last_user_msg = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"),
            ""
        )
        return f"[模拟回复] 基于历史 ({len(messages)}条) 回复: {last_user_msg[:40]}..."
    
    client = OpenAI(api_key=api_key)
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        max_tokens=512,
    )
    
    return response.choices[0].message.content or ""


# 演示：Few-shot 分类
print("=" * 60)
print("Few-shot 分类示例")
print("=" * 60)

classification_messages = [
    {"role": "system", "content": "你是一个客服工单分类器。请将工单归类为: bug, feature, question。"},
    # Few-shot 示例
    {"role": "user", "content": "登录页面点击按钮没反应"},
    {"role": "assistant", "content": "bug"},
    {"role": "user", "content": "能不能加一个导出PDF的功能？"},
    {"role": "assistant", "content": "feature"},
    # 真正的输入
    {"role": "user", "content": "如何修改我的账户邮箱地址？"},
]

category = multi_turn_chat(classification_messages)
print(f"  分类结果: {category}")
print(f"  对话轮数: {len(classification_messages)}（含 {3} 个用户问题）")

### ✏️ 练习 1: OpenAI 基础调用

1. 修改 `basic_chat_completion` 使其支持 `stop` 参数（例如遇到 "。" 就停止）。
2. 实现一个 `compare_models(prompt, models)` 函数，同一 prompt 发给多个模型（如 gpt-4o 和 gpt-4o-mini），对比回复和成本。
3. 使用 `response.choices[0].message.tool_calls` 探索 function calling 的响应结构。

---
## 2. Streaming 流式响应

### 为什么要学？
流式输出是生产级 LLM 应用的标配。用户不愿意等 10 秒才看到完整回复，
流式输出可以逐 token 显示，大幅提升用户体验（TTFB 从 10s 降到 ~0.5s）。

在 LangChain / LangGraph 中，流式输出通过 `astream_events` 或自定义 callback 实现，
底层原理就是本节讲的 `stream=True`。

In [ ]:
"""
Streaming 流式响应 —— Token by Token
======================================

OpenAI Python SDK 支持同步和异步两种流式模式：
- stream=True → 返回一个迭代器（同步）
- 异步迭代（async for chunk in stream）

本 Cell 展示同步流式，异步版见 Cell 7 综合实战。
"""
from typing import Iterator, Generator
import time

# ============================================================
# Streaming 基础
# ============================================================

def stream_chat_completion(
    prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    max_tokens: int = 256,
) -> Generator[Dict[str, Any], None, None]:
    """
    流式 Chat Completion —— 使用 Generator 逐步 yield token。
    
    生成器函数特点：
    - 每次 yield 一个 chunk，调用方立即处理
    - 不需要等整个响应完成
    - 内存友好（不需要缓存完整响应）
    
    Yields:
        dict: {
            "delta": "当前 token 文本",
            "finish_reason": None | "stop" | "length",
            "index": choice 索引,
        }
    """
    if not HAS_REAL_KEY or not OPENAI_AVAILABLE:
        # 模拟流式输出
        mock_response = "这是模拟的流式输出。每个字符将逐个发送。"
        for i, char in enumerate(mock_response):
            time.sleep(0.03)  # 模拟流式延迟
            yield {
                "delta": char,
                "finish_reason": "stop" if i == len(mock_response) - 1 else None,
                "index": 0,
            }
        return
    
    client = OpenAI(api_key=api_key)
    
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "你是一个有帮助的助手。"},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        stream=True,  # ← 关键参数
    )
    
    for chunk in stream:
        # OpenAI streaming chunks 结构：
        # chunk.choices[0].delta.content → 当前 token 的文本
        # chunk.choices[0].finish_reason → None 或 "stop"
        if chunk.choices and len(chunk.choices) > 0:
            delta = chunk.choices[0].delta
            yield {
                "delta": delta.content or "",
                "finish_reason": chunk.choices[0].finish_reason,
                "index": chunk.choices[0].index,
            }


# 演示：流式输出
print("=" * 60)
print("流式输出演示")
print("=" * 60)
print("Prompt: '请用三句话介绍 Python。'")
print()

full_content = ""
token_count = 0
start_time = time.perf_counter()

print("📨 流式输出: ", end="", flush=True)
for chunk in stream_chat_completion(
    prompt="请用三句话介绍 Python。",
    max_tokens=150,
):
    delta = chunk["delta"]
    if delta:
        print(delta, end="", flush=True)  # 立即打印每个 token
        full_content += delta
        token_count += 1
    
    if chunk["finish_reason"]:
        print(f"\n  [{chunk['finish_reason']}]", end="")

elapsed = time.perf_counter() - start_time
print()
print()
print(f"  总 tokens: ~{token_count}")
print(f"  总耗时: {elapsed:.2f}s")
print(f"  完整内容 ({len(full_content)} 字): {full_content[:80]}...")

In [ ]:
# ============================================================
# 流式输出的高级用法：带状态回调
# ============================================================

from enum import Enum

class StreamEvent(Enum):
    """流式事件类型"""
    TOKEN = "token"           # 收到一个 token
    THINKING = "thinking"     # 思考阶段（部分模型支持）
    TOOL_CALL = "tool_call"   # 工具调用（function calling）
    DONE = "done"             # 流结束
    ERROR = "error"           # 发生错误

def stream_with_events(
    prompt: str,
    on_token: Optional[callable] = None,
    on_done: Optional[callable] = None,
) -> str:
    """
    带事件回调的流式调用。
    
    这是 LangChain Callback 系统的简化模拟：
    - on_token: 每收到 token 时调用
    - on_done: 流结束时调用（传入完整响应）
    
    在实际项目中，这些回调可用于：
    - 更新 UI（WebSocket 推送每个 token）
    - 记录日志
    - 实时计算 token 成本
    """
    full_response = ""
    
    for chunk in stream_chat_completion(prompt=prompt, max_tokens=200):
        delta = chunk["delta"]
        if delta:
            full_response += delta
            if on_token:
                on_token(delta, len(full_response))
        
        if chunk["finish_reason"]:
            if on_done:
                on_done(full_response, chunk["finish_reason"])
            break
    
    return full_response


# 演示：实时仪表盘风格的流式输出
print("=" * 60)
print("实时 Token 计数器演示")
print("=" * 60)
print()

# 回调函数：模拟 WebSocket 推送或 UI 更新
def live_token_counter(token: str, total: int) -> None:
    """每收到一个 token 时更新计数器"""
    # 在实际应用中，这里是 WebSocket.send() 或 streamlit.write()
    if total % 5 == 0:  # 每 5 个 token 打印一次进度
        print(f"\r  📊 已收到 {total} tokens...", end="", flush=True)

def on_stream_done(full_text: str, reason: str) -> None:
    """流完成时输出摘要"""
    print(f"\n  ✅ 完成 (finish_reason={reason}), 共 {len(full_text)} 字符")

response = stream_with_events(
    prompt="什么是RAG（检索增强生成）？",
    on_token=live_token_counter,
    on_done=on_stream_done,
)

print()
print(f"  完整响应: {response[:100]}...")

### ✏️ 练习 2: Streaming

1. 实现一个 `async_stream_chat_completion` 异步版本（使用 `async for`）。
2. 添加缓冲机制：收集前 50 个 token 到缓冲区，之后每 10 个 token 批量推送给 UI。
3. 模拟中断（用户点击 Stop 按钮）：在流式循环中检测一个 `stop_event` 标志。

---
## 3. Token 计数与预算管理

### 为什么要学？
Token 是 LLM 的成本单位和上下文窗口单位。精确的 Token 计数让你：
- 在发送请求前检查是否超出模型的上下文窗口
- 估算每轮对话的成本
- 决定是否需要截断（truncate）或分块（chunking）

`tiktoken` 是 OpenAI 的官方 tokenizer，速度极快（纯 Rust 实现）。

In [ ]:
"""
Token 计数与预算管理
=====================

核心工具：tiktoken（OpenAI 官方，~100x 比 transformers 快）
"""

try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
    print("✅ tiktoken 已安装")
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("⚠️  tiktoken 未安装。运行: pip install tiktoken")
    print("   将使用字符数 / 4 的简单估算。")

# ============================================================
# 获取模型对应的 tokenizer
# ============================================================

# 常见模型 → encoding 映射表
MODEL_ENCODING_MAP = {
    # GPT-4 系列 (o200k_base)
    "gpt-4o": "o200k_base",
    "gpt-4o-mini": "o200k_base",
    "gpt-4-turbo": "cl100k_base",
    "gpt-4": "cl100k_base",
    # GPT-3.5 系列 (cl100k_base)
    "gpt-3.5-turbo": "cl100k_base",
    # Embedding (cl100k_base)
    "text-embedding-3-small": "cl100k_base",
    "text-embedding-ada-002": "cl100k_base",
}

def get_tokenizer(model: str = "gpt-4o"):
    """
    获取模型对应的 tiktoken encoding。
    
    为什么每个模型有不同 encoding：
    - 不同模型的 tokenizer 不同（BPE 词汇表大小不同）
    - 用错 encoding 会导致 token 计数不准
    - gpt-4o 使用 o200k_base（200k 词汇表）
    - gpt-4/3.5 使用 cl100k_base（100k 词汇表）
    """
    if not TIKTOKEN_AVAILABLE:
        return None
    
    encoding_name = MODEL_ENCODING_MAP.get(model, "cl100k_base")
    try:
        return tiktoken.get_encoding(encoding_name)
    except Exception:
        return tiktoken.get_encoding("cl100k_base")  # 回退


# ============================================================
# Token 计数函数
# ============================================================

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """
    计算文本的 token 数量。
    
    Args:
        text: 要计数的文本
        model: 模型名称（决定使用哪个 tokenizer）
    
    Returns:
        token 数量
    
    使用场景：
    - 请求前：检查 prompt 是否超出上下文窗口
    - 请求后：计算实际消耗
    - 成本估算：tokens * price_per_token
    """
    if TIKTOKEN_AVAILABLE:
        enc = get_tokenizer(model)
        if enc:
            return len(enc.encode(text))
    # 回退：简单估算（约 4 字符 = 1 token，中文约 1-2 字符 = 1 token）
    return len(text) // 3


def count_messages_tokens(
    messages: List[Dict[str, str]],
    model: str = "gpt-4o",
) -> int:
    """
    计算消息列表的总 token 数（含消息格式开销）。
    
    每条消息有固定的格式开销（约 3-4 tokens），
    OpenAI 官方推荐在内容 tokens 的基础上加这个开销。
    
    Args:
        messages: 消息列表 [{role, content}, ...]
        model: 模型名称
    
    Returns:
        总 token 数（含格式开销）
    """
    total = 0
    for msg in messages:
        total += count_tokens(msg.get("content", ""), model)
        total += count_tokens(msg.get("role", ""), model)
        total += 3  # 每条消息约 3 tokens 格式开销
    total += 3  # 回复的 priming token
    return total


# 演示
test_messages = [
    {"role": "system", "content": "你是一个有帮助的助手。请用中文回答。"},
    {"role": "user", "content": "请详细解释什么是向量数据库，以及它与传统数据库的区别。"},
]

print("=" * 60)
print("Token 计数演示")
print("=" * 60)

for msg in test_messages:
    role = msg["role"]
    content = msg["content"]
    tokens = count_tokens(content)
    print(f"  [{role}] {content[:40]}... → {tokens} tokens")

total = count_messages_tokens(test_messages)
print(f"  总计（含格式开销）: {total} tokens")
print()

# 中文 Token 化特点演示
chinese_text = "人工智能正在改变世界"
english_text = "Artificial Intelligence is changing the world"
print(f"  中文: '{chinese_text}' → {count_tokens(chinese_text)} tokens")
print(f"  英文: '{english_text}' → {count_tokens(english_text)} tokens")
print(f"  💡 中文每个字约占 1-2 tokens，英文每个词约占 1-2 tokens")

In [ ]:
# ============================================================
# Token 预算管理 —— 生产级模式
# ============================================================

# 常见模型的上下文窗口
MODEL_CONTEXT_WINDOWS = {
    "gpt-4o": 128_000,
    "gpt-4o-mini": 128_000,
    "gpt-4-turbo": 128_000,
    "gpt-4": 8_192,
    "gpt-3.5-turbo": 16_385,
    "claude-sonnet-4-6-20250514": 200_000,
    "claude-3-opus-20240229": 200_000,
    "claude-3-haiku-20240307": 200_000,
}

class TokenBudgetExceededError(Exception):
    """Token 预算超限异常"""
    pass


class TokenBudget:
    """
    Token 预算管理器。
    
    功能：
    - 追踪已使用的 token
    - 请求前验证是否超出上下文窗口
    - 预留输出 token 空间
    
    典型用法：
    ```python
    budget = TokenBudget("gpt-4o", max_output_tokens=4096)
    messages = build_messages(user_input, history)
    
    # 自动截断超出预算的历史消息
    messages = budget.fit_to_budget(messages)
    
    # 发送请求后再更新预算
    budget.update(response.usage.total_tokens)
    ```
    """
    
    def __init__(
        self,
        model: str = "gpt-4o",
        max_output_tokens: int = 4096,
        safety_margin: float = 0.1,  # 10% 安全余量
    ):
        self.model = model
        self.context_window = MODEL_CONTEXT_WINDOWS.get(model, 128_000)
        self.max_output_tokens = max_output_tokens
        self.safety_margin = safety_margin
        self.total_used = 0
        self.call_count = 0
    
    @property
    def max_input_tokens(self) -> int:
        """最大允许的输入 token 数（含安全余量）"""
        effective_limit = self.context_window - self.max_output_tokens
        return int(effective_limit * (1 - self.safety_margin))
    
    @property
    def remaining(self) -> int:
        """剩余的 token 预算"""
        return self.max_input_tokens - self.total_used
    
    def check_budget(self, token_count: int) -> bool:
        """
        检查是否有足够的 token 预算。
        
        Args:
            token_count: 即将使用的 token 数
        
        Returns:
            True 如果预算充足
        
        Raises:
            TokenBudgetExceededError: 如果预算不足
        """
        if self.total_used + token_count > self.max_input_tokens:
            raise TokenBudgetExceededError(
                f"Token 预算超限: 需要 {token_count}, "
                f"剩余 {self.remaining}, "
                f"上限 {self.max_input_tokens}"
            )
        return True
    
    def update(self, tokens_used: int) -> None:
        """
        更新已使用的 token 计数。
        每次 API 调用后调用此方法。
        """
        self.total_used += tokens_used
        self.call_count += 1
    
    def fit_to_budget(
        self,
        messages: List[Dict[str, str]],
    ) -> List[Dict[str, str]]:
        """
        如果消息超出预算，从最早的消息开始截断。
        
        策略：保留 system message，从最早的 user/assistant 开始删除。
        这是滑动窗口（sliding window）策略的实现。
        """
        total = count_messages_tokens(messages, self.model)
        if total <= self.max_input_tokens:
            return messages
        
        # 找到 system message 的索引
        system_idx = next(
            (i for i, m in enumerate(messages) if m["role"] == "system"),
            None
        )
        
        # 从后往前保留，始终保留 system message
        fitted = []
        if system_idx is not None:
            fitted.append(messages[system_idx])
        
        current_tokens = count_messages_tokens(fitted, self.model)
        
        for msg in reversed(messages):
            if msg["role"] == "system":
                continue  # 已经添加过了
            msg_tokens = count_tokens(msg["content"], self.model) + 4
            if current_tokens + msg_tokens > self.max_input_tokens:
                break
            fitted.insert(1 if system_idx is not None else 0, msg)
            current_tokens += msg_tokens
        
        print(f"  ⚠️  Token 预算截断: {len(messages)} → {len(fitted)} 条消息")
        return fitted
    
    def report(self) -> str:
        """生成预算报告"""
        usage_pct = (self.total_used / self.max_input_tokens * 100) if self.max_input_tokens else 0
        return (
            f"Token 预算: {self.total_used:,}/{self.max_input_tokens:,} "
            f"({usage_pct:.1f}%) | 调用次数: {self.call_count}"
        )


# 演示
print("=" * 60)
print("Token 预算管理演示")
print("=" * 60)

budget = TokenBudget(model="gpt-4o", max_output_tokens=4096)
print(f"  模型: gpt-4o")
print(f"  上下文窗口: {budget.context_window:,}")
print(f"  最大输入: {budget.max_input_tokens:,} (含 {budget.safety_margin:.0%} 安全余量)")
print(f"  预留输出: {budget.max_output_tokens:,}")
print(f"  剩余: {budget.remaining:,}")

# 模拟一次调用
mock_token_usage = 1500
budget.check_budget(mock_token_usage)  # 应该通过
budget.update(mock_token_usage)
print(f"\n  调用后: {budget.report()}")

# 模拟超出预算
print()
try:
    budget.check_budget(200_000)  # 远超出
except TokenBudgetExceededError as e:
    print(f"  ❌ {e}")

print()
print("💡 生产建议: 每次 LLM 调用前都用 TokenBudget.check_budget() 验证")

### ✏️ 练习 3: Token 管理

1. 扩展 `TokenBudget` 使其支持多个 Provider（Anthropic 也用不同 tokenizer）。
2. 实现 `summarize_history(messages, max_tokens)` 函数：当历史消息超出预算时，用 LLM 生成历史摘要代替原始消息。
3. 编写单元测试验证 `fit_to_budget` 在边界情况下的正确性。

---
## 4. 错误处理与生产级重试

### 为什么要学？
LLM API 调用在生产中会遇到各种错误：认证失败(401)、限流(429)、
服务器错误(500)、服务过载(503)。**绝不能**让这些错误直接崩溃你的应用。

关键策略：
- 区分可重试错误和不可重试错误
- 指数退避 + 随机抖动（避免雷鸣羊群效应）
- 使用全 jitter 算法而非固定等待

注意：本课程故意不使用 tenacity 库，而是手动实现重试逻辑，
让你深入理解底层机制。

In [ ]:
"""
错误处理 —— 区分可重试与不可重试
==================================

HTTP 状态码分类：
- 401 Unauthorized: 不可重试（API Key 问题）
- 429 Rate Limit: 可重试（等一会再试）
- 500 Internal Server Error: 可重试（临时故障）
- 503 Service Unavailable: 可重试（服务过载）
- 400 Bad Request: 不可重试（请求格式问题）
"""
import random
import math
from enum import Enum


class ErrorCategory(Enum):
    """错误类别"""
    RETRYABLE = "retryable"        # 可重试
    NON_RETRYABLE = "non_retryable" # 不可重试
    AUTH = "auth"                   # 认证错误（特殊处理）


def categorize_error(status_code: int, error_type: str = "") -> ErrorCategory:
    """
    根据 HTTP 状态码和异常类型分类错误。
    
    Args:
        status_code: HTTP 状态码
        error_type: 异常类型名称（用于 OpenAI SDK 的异常分类）
    
    Returns:
        ErrorCategory 枚举值
    """
    # 认证相关：401, 403
    if status_code in (401, 403):
        return ErrorCategory.AUTH
    
    # 可重试：429（限流）, 500（服务器错误）, 502, 503, 504
    if status_code in (429, 500, 502, 503, 504):
        return ErrorCategory.RETRYABLE
    
    # OpenAI SDK 特定的可重试异常
    if error_type in ("RateLimitError", "APITimeoutError", "APIConnectionError",
                       "InternalServerError", "ServiceUnavailableError"):
        return ErrorCategory.RETRYABLE
    
    # 其他：不可重试
    return ErrorCategory.NON_RETRYABLE


# 演示
test_cases = [
    (401, "AuthenticationError"),
    (429, "RateLimitError"),
    (500, "InternalServerError"),
    (503, "ServiceUnavailableError"),
    (400, "BadRequestError"),
]

print("=" * 60)
print("错误分类演示")
print("=" * 60)
for code, err_type in test_cases:
    cat = categorize_error(code, err_type)
    emoji = {ErrorCategory.RETRYABLE: "🔄", ErrorCategory.AUTH: "🔑", ErrorCategory.NON_RETRYABLE: "🚫"}
    print(f"  {emoji[cat]} HTTP {code} ({err_type}) → {cat.value}")

In [ ]:
# ============================================================
# 指数退避 + 抖动（Exponential Backoff + Jitter）
# ============================================================

def calculate_backoff(
    attempt: int,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    backoff_factor: float = 2.0,
    jitter: bool = True,
) -> float:
    """
    计算指数退避等待时间（带抖动）。
    
    三种抖动策略：
    1. Full Jitter (推荐): delay = random(0, exponential_backoff)
    2. Equal Jitter: delay = exp/2 + random(0, exp/2)
    3. Decorrelated Jitter: delay = min(max, random(base, delay*3))
    
    为什么需要抖动（Jitter）？
    - 避免"雷鸣羊群"效应（Thundering Herd）
    - 当大量客户端同时遇到 429，如果不加抖动，
      所有客户端会在同一时刻重试，导致新一轮限流
    - 加随机抖动后，重试时间分散开，服务器压力平滑
    
    Args:
        attempt: 当前尝试次数（从 0 开始）
        base_delay: 基础等待时间（秒）
        max_delay: 最大等待时间（秒）
        backoff_factor: 退避因子（通常为 2）
        jitter: 是否启用抖动
    
    Returns:
        等待时间（秒）
    
    Examples:
        attempt=0: ~0.5-1s   (第一次重试很快)
        attempt=1: ~1-2s
        attempt=2: ~2-4s
        attempt=3: ~4-8s
        attempt=4: ~8-16s
        attempt=5: capped at 60s
    """
    exponential = base_delay * (backoff_factor ** attempt)
    
    if jitter:
        # Full Jitter: 在 [0, exponential) 之间均匀随机
        # 这是 AWS Architecture Blog 推荐的方式
        delay = random.uniform(0, exponential)
    else:
        delay = exponential
    
    return min(delay, max_delay)


def demo_backoff_schedule():
    """演示不同退避策略的差异"""
    print("=" * 60)
    print("指数退避 + 抖动 调度表")
    print("=" * 60)
    print(f"{'尝试':<8} {'无抖动(s)':<12} {'Full Jitter(s)':<18} {'Equal Jitter(s)':<18}")
    print("-" * 56)
    
    random.seed(42)  # 固定种子以便可复现
    
    for attempt in range(8):
        no_jitter = calculate_backoff(attempt, jitter=False)
        full_jitter = calculate_backoff(attempt, jitter=True)
        
        # Equal Jitter: exp/2 + random(0, exp/2)
        exp = 1.0 * (2.0 ** attempt)
        equal_jitter = exp / 2 + random.uniform(0, exp / 2)
        
        print(f"  {attempt:<8} {no_jitter:<12.2f} {full_jitter:<18.2f} {min(equal_jitter, 60):<18.2f}")
    
    print()
    print("💡 Full Jitter 的值每次不同（随机），可避免雷鸣羊群效应")

demo_backoff_schedule()

In [ ]:
# ============================================================
# 完整的重试包装器（不依赖 tenacity）
# ============================================================

import asyncio
import functools

class RetryConfig:
    """
    重试配置。
    
    集中管理所有重试参数，方便在不同 LLM Provider 间复用。
    """
    def __init__(
        self,
        max_retries: int = 3,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        backoff_factor: float = 2.0,
        jitter: bool = True,
        retryable_statuses: tuple = (429, 500, 502, 503, 504),
    ):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.backoff_factor = backoff_factor
        self.jitter = jitter
        self.retryable_statuses = retryable_statuses


# 常量配置（生产环境建议）
DEFAULT_RETRY_CONFIG = RetryConfig(
    max_retries=3,
    base_delay=1.0,
    max_delay=30.0,
)

AGGRESSIVE_RETRY_CONFIG = RetryConfig(
    max_retries=5,
    base_delay=2.0,
    max_delay=120.0,  # 更长的等待时间
)


async def retry_with_backoff(
    coro_func,
    *args,
    retry_config: Optional[RetryConfig] = None,
    **kwargs,
) -> Any:
    """
    带指数退避 + 抖动的异步重试包装器。
    
    为什么自己实现而不是用 tenacity？
    - 深入理解重试机制（课程目标）
    - 对异步 Python 的更细粒度控制
    - 自定义错误分类逻辑
    - 无外部依赖
    
    Args:
        coro_func: 异步函数（将被调用）
        *args: 位置参数
        retry_config: 重试配置（可选，默认 DEFAULT_RETRY_CONFIG）
        **kwargs: 关键字参数
    
    Returns:
        协程的返回值
    
    Raises:
        原始异常（重试耗尽后）
    
    Usage:
        result = await retry_with_backoff(
            client.chat.completions.create,
            model="gpt-4o",
            messages=[...],
        )
    """
    if retry_config is None:
        retry_config = DEFAULT_RETRY_CONFIG
    
    last_exception = None
    
    for attempt in range(retry_config.max_retries + 1):  # +1 包含首次尝试
        try:
            return await coro_func(*args, **kwargs)
        except Exception as e:
            last_exception = e
            
            # 提取 HTTP 状态码（兼容 OpenAI 和 Anthropic SDK）
            status_code = getattr(e, "status_code", 0)
            if status_code == 0:
                status_code = getattr(getattr(e, "response", None), "status_code", 0) or 0
            
            error_type = type(e).__name__
            category = categorize_error(status_code, error_type)
            
            # 不可重试错误 → 立即抛出
            if category == ErrorCategory.NON_RETRYABLE:
                print(f"  🚫 不可重试错误: HTTP {status_code} ({error_type})")
                raise
            
            # 认证错误 → 立即抛出（附带明确提示）
            if category == ErrorCategory.AUTH:
                print(f"  🔑 认证失败: HTTP {status_code} ({error_type})")
                print(f"     请检查 API Key 是否正确设置。")
                raise
            
            # 可重试错误 —— 但重试次数已用完
            if attempt >= retry_config.max_retries:
                print(f"  ❌ 已重试 {retry_config.max_retries} 次，仍然失败")
                raise
            
            # 可重试错误 —— 计算等待时间并重试
            delay = calculate_backoff(
                attempt=attempt,
                base_delay=retry_config.base_delay,
                max_delay=retry_config.max_delay,
                backoff_factor=retry_config.backoff_factor,
                jitter=retry_config.jitter,
            )
            
            print(
                f"  🔄 重试 {attempt + 1}/{retry_config.max_retries}: "
                f"HTTP {status_code} ({error_type}), "
                f"等待 {delay:.1f}s..."
            )
            
            await asyncio.sleep(delay)
    
    # 理论上不会到这里（因为上面会 raise）
    raise last_exception  # type: ignore


# 演示：模拟重试过程
async def demo_retry():
    """演示完整的重试流程（用模拟 API 调用）"""
    print("=" * 60)
    print("模拟重试流程演示")
    print("=" * 60)
    print()
    
    call_count = [0]  # 用 list 来在闭包中修改
    
    async def mock_api_call(fail_times: int = 2):
        """
        模拟 API 调用：前 fail_times 次返回 429，之后成功。
        """
        call_count[0] += 1
        if call_count[0] <= fail_times:
            # 构造一个模拟的 RateLimitError
            class MockRateLimitError(Exception):
                def __init__(self):
                    self.status_code = 429
            raise MockRateLimitError()
        return {"choices": [{"message": {"content": "✅ 调用成功！"}}]}
    
    # 使用我们的重试包装器
    try:
        result = await retry_with_backoff(
            mock_api_call,
            fail_times=2,  # 前 2 次失败，第 3 次成功
            retry_config=DEFAULT_RETRY_CONFIG,
        )
        print(f"\n  ✅ 最终结果: {result['choices'][0]['message']['content']}")
        print(f"  总调用次数: {call_count[0]}")
    except Exception as e:
        print(f"  ❌ 最终失败: {e}")
    
    print()
    
    # 演示：不可重试错误立即抛出
    print("演示不可重试错误:")
    async def mock_auth_error():
        class MockAuthError(Exception):
            def __init__(self):
                self.status_code = 401
        raise MockAuthError()
    
    try:
        await retry_with_backoff(mock_auth_error)
    except Exception:
        print("  ✅ 401 认证错误立即抛出，不重试（正确行为）")

asyncio.run(demo_retry())

### ✏️ 练习 4: 错误处理与重试

1. 为 `retry_with_backoff` 添加 `on_retry` 回调参数，让调用方可以记录重试日志。
2. 实现 `circuit_breaker` 模式：连续失败 N 次后，停止请求一段时间（熔断）。
3. 对比 Full Jitter vs Equal Jitter 在 100 个并发客户端的重试时间分布。

---
## 5. Anthropic Messages API

### 为什么要学？
Claude 系列模型在长文本理解、指令遵循、安全性方面有独特优势。
本课程的 Agent 和 RAG 系统会同时使用 OpenAI 和 Anthropic 作为 LLM 后端，
掌握两个 Provider 的 API 模式是必备技能。

Anthropic Messages API 与 OpenAI Chat Completions 的**关键差异**：
- 无独立的 `system` role —— 系统提示词是顶级参数
- `max_tokens` 是**必填参数**（Anthropic 的安全设计）
- 默认支持更大的上下文窗口（200K）

In [ ]:
"""
Anthropic Messages API —— 等效模式
=====================================

前置条件：设置环境变量 ANTHROPIC_API_KEY
  export ANTHROPIC_API_KEY="sk-ant-..."
"""

# 尝试导入 Anthropic SDK
try:
    import anthropic
    from anthropic.types import Message
    ANTHROPIC_AVAILABLE = True
    print("✅ Anthropic SDK 已安装")
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️  Anthropic SDK 未安装。运行: pip install anthropic")
    print("   将使用模拟输出演示 API 用法。")

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY", "")
HAS_ANTHROPIC_KEY = bool(anthropic_api_key and anthropic_api_key.startswith("sk-ant-"))

if not HAS_ANTHROPIC_KEY:
    print("ℹ️  未设置有效的 ANTHROPIC_API_KEY，将使用模拟输出。")

print()

In [ ]:
# ============================================================
# Anthropic Messages API —— 基本调用
# ============================================================

def claude_chat_completion(
    prompt: str,
    model: str = "claude-sonnet-4-6-20250514",
    system_prompt: str = "你是一个有帮助的AI助手。请用中文回答。",
    temperature: float = 0.0,
    max_tokens: int = 256,
) -> Dict[str, Any]:
    """
    Anthropic Messages API 调用封装。
    
    与 OpenAI 的关键差异：
    1. system 是顶级参数，不是 messages 中的一条
    2. max_tokens 是必填参数
    3. 返回结构不同: response.content[0].text
    4. 没有 seed 参数，但 temperature=0 同样确定性
    5. 支持 stop_sequences 参数
    
    Args:
        prompt: 用户输入
        model: Claude 模型 ID
        system_prompt: 系统提示词（顶级参数，非 message）
        temperature: 温度（0.0-1.0）
        max_tokens: 最大输出 token（必填！）
    
    Returns:
        与 basic_chat_completion 相同格式的字典，便于 Provider 切换
    """
    if not HAS_ANTHROPIC_KEY or not ANTHROPIC_AVAILABLE:
        return {
            "content": f"[模拟 Claude] 关于 '{prompt[:50]}...' 的回答",
            "model": model,
            "usage": {"input_tokens": 15, "output_tokens": 25, "total_tokens": 40},
            "finish_reason": "end_turn",
        }
    
    client = anthropic.Anthropic(api_key=anthropic_api_key)
    
    response: Message = client.messages.create(
        model=model,
        system=system_prompt,  # ← Anthropic: 顶级参数
        messages=[
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,  # ← Anthropic: 必填
    )
    
    return {
        "content": response.content[0].text if response.content else "",
        "model": response.model,
        "usage": {
            "input_tokens": response.usage.input_tokens if response.usage else 0,
            "output_tokens": response.usage.output_tokens if response.usage else 0,
            "total_tokens": (
                (response.usage.input_tokens or 0) + (response.usage.output_tokens or 0)
                if response.usage else 0
            ),
        },
        "finish_reason": response.stop_reason or "unknown",
    }


# 演示
print("=" * 60)
print("Claude Messages API 基本调用")
print("=" * 60)

result = claude_chat_completion(
    prompt="什么是Agent（智能体）？请用一句话概括。",
    max_tokens=200,
)
print(f"  模型: {result['model']}")
print(f"  回复: {result['content'][:100]}...")
print(f"  tokens: {result['usage']['total_tokens']}")
print(f"  停止原因: {result['finish_reason']}")
print()

# 注意 Anthropic 的 stop_reason 值：
# - "end_turn": 模型自然结束（等同于 OpenAI "stop"）
# - "max_tokens": 达到 max_tokens 上限（等同于 OpenAI "length"）
# - "stop_sequence": 遇到 stop_sequences 中的停止词
# - "tool_use": 模型调用了工具

In [ ]:
# ============================================================
# Anthropic Streaming
# ============================================================

def claude_stream_chat(
    prompt: str,
    model: str = "claude-sonnet-4-6-20250514",
    max_tokens: int = 256,
):
    """
    Claude 流式响应。
    
    与 OpenAI streaming 的关键差异：
    - Anthropic 使用事件类型 (event.type) 来区分不同 chunk
    - 主要事件类型:
      - "content_block_delta": 文本增量（类似 OpenAI delta）
      - "content_block_start": 内容块开始
      - "content_block_stop": 内容块结束
      - "message_delta": 消息元数据更新（含 stop_reason 和 usage）
      - "message_stop": 消息结束
    - text delta 在 event.delta.text 中
    """
    if not HAS_ANTHROPIC_KEY or not ANTHROPIC_AVAILABLE:
        # 模拟流式输出
        mock = "这是 Claude 模拟的流式回复演示文本。每个字符依次输出。"
        for char in mock:
            yield {"delta": char, "type": "content_block_delta"}
        yield {"delta": "", "type": "message_stop", "stop_reason": "end_turn"}
        return
    
    client = anthropic.Anthropic(api_key=anthropic_api_key)
    
    with client.messages.stream(
        model=model,
        system="你是一个有帮助的助手。请用中文回答。",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
    ) as stream:
        for event in stream:
            event_type = getattr(event, "type", "unknown")
            
            if event_type == "content_block_delta":
                delta_text = getattr(event.delta, "text", "")
                yield {"delta": delta_text, "type": event_type}
            
            elif event_type == "message_delta":
                yield {
                    "delta": "",
                    "type": event_type,
                    "stop_reason": getattr(event.delta, "stop_reason", None),
                }
            
            elif event_type == "message_stop":
                yield {"delta": "", "type": event_type}


# 演示
print("=" * 60)
print("Claude 流式输出")
print("=" * 60)
print("Prompt: '什么是RAG（检索增强生成）？'")
print()

full_text = ""
print("📨 Claude 回复: ", end="", flush=True)

for chunk in claude_stream_chat(
    prompt="什么是RAG（检索增强生成）？请用三句话解释。",
    max_tokens=200,
):
    if chunk["type"] == "content_block_delta" and chunk["delta"]:
        print(chunk["delta"], end="", flush=True)
        full_text += chunk["delta"]
    elif chunk["type"] == "message_delta":
        print(f"\n  [stop_reason: {chunk.get('stop_reason')}]")

print(f"\n  ✅ 完整回复 ({len(full_text)} 字)")

### ✏️ 练习 5: Anthropic API

1. 实现 `claude_with_tool_use(prompt, tools)` 函数，支持 Anthropic 的 tool_use 功能（返回 tool_use block）。
2. 对比 Anthropic 的 system prompt 机制与 OpenAI 的 system message 在 token 计数上的差异。
3. 测试 Anthropic 的长上下文能力：发送一个 50K token 的 prompt，观察响应质量。

---
## 6. Provider 对比：同一 Prompt 看差异

### 为什么要学？
不同 LLM 对同一 prompt 的回复风格、格式、详细程度差异显著。
在实际项目中，你可能需要：
- 根据任务类型选择合适的模型（代码用 Claude，闲聊用 GPT-4o）
- 设计 prompt 时要考虑不同模型的特性
- 比较成本与质量

In [ ]:
"""
Provider 对比 —— 同一 Prompt, 不同模型
========================================

本 Cell 对比 OpenAI 和 Anthropic 对相同 prompt 的响应差异。
注意：这需要两个 API Key 都设置好才能真实对比。
否则会使用模拟输出。
"""
import asyncio
import time


async def compare_providers(
    prompt: str,
    openai_model: str = "gpt-4o-mini",
    anthropic_model: str = "claude-sonnet-4-6-20250514",
) -> Dict[str, Any]:
    """
    并发调用两个 Provider，对比结果。
    
    对比维度：
    - 响应内容
    - 响应速度（延迟）
    - Token 消耗
    - 回复风格
    
    使用 asyncio.gather 并发调用以获得公平的速度对比。
    """
    async def call_openai():
        start = time.perf_counter()
        result = basic_chat_completion(
            prompt=prompt,
            model=openai_model,
            temperature=0.0,
            max_tokens=256,
        )
        elapsed = time.perf_counter() - start
        result["latency_s"] = elapsed
        return result
    
    async def call_anthropic():
        start = time.perf_counter()
        result = claude_chat_completion(
            prompt=prompt,
            model=anthropic_model,
            temperature=0.0,
            max_tokens=256,
        )
        elapsed = time.perf_counter() - start
        result["latency_s"] = elapsed
        return result
    
    # 并发调用
    openai_result, anthropic_result = await asyncio.gather(
        call_openai(),
        call_anthropic(),
    )
    
    return {
        "prompt": prompt,
        "openai": openai_result,
        "anthropic": anthropic_result,
    }


# 运行对比
async def run_comparison():
    print("=" * 70)
    print("Provider 对比：同一 Prompt → 不同 LLM")
    print("=" * 70)
    print()
    
    test_prompts = [
        "用一句话解释什么是机器学习。",
        "写一个 Python 函数，计算两个列表的 Jaccard 相似度。",
    ]
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"--- Prompt {i}: {prompt[:60]}... ---")
        print()
        
        comparison = await compare_providers(prompt)
        
        # OpenAI 结果
        oai = comparison["openai"]
        print(f"  🤖 OpenAI ({oai['model']}):")
        print(f"     回复: {oai['content'][:100]}...")
        print(f"     耗时: {oai['latency_s']:.2f}s")
        print(f"     Tokens: {oai['usage'].get('total_tokens', 'N/A')}")
        print()
        
        # Anthropic 结果
        ant = comparison["anthropic"]
        print(f"  🧠 Anthropic ({ant['model']}):")
        print(f"     回复: {ant['content'][:100]}...")
        print(f"     耗时: {ant['latency_s']:.2f}s")
        print(f"     Tokens: {ant['usage'].get('total_tokens', 'N/A')}")
        print()
        
        # 简单对比
        oai_len = len(oai['content'])
        ant_len = len(ant['content'])
        print(f"  📊 字符数: OpenAI={oai_len}, Anthropic={ant_len}")
        print(f"  📊 速度比: {'OpenAI 更快' if oai['latency_s'] < ant['latency_s'] else 'Anthropic 更快'}")
        print()

asyncio.run(run_comparison())

print("💡 实际对比时注意：")
print("  - 价格: GPT-4o-mini < Claude Haiku < GPT-4o < Claude Sonnet")
print("  - 代码能力: Claude (Sonnet/Opus) 通常更优")
print("  - 指令遵循: Claude 更严格")
print("  - 中文: 两者都很好，Claude 有时更自然")
print("  - 上下文窗口: Claude 200K > GPT-4o 128K")

### ✏️ 练习 6: Provider 对比

1. 扩展对比维度：添加 `response.usage` 的成本计算（见下一节）。
2. 添加更多 Provider：Gemini、DeepSeek、Qwen 等。
3. 设计一个统一的 `LLMProvider` Protocol 抽象接口，使得切换 Provider 只需改一行配置。

---
## 7. 成本追踪（Cost Tracking）

### 为什么要学？
LLM API 调用是按量计费的。生产级应用需要精确追踪每次调用、
每个会话、每个用户的成本。没有成本追踪 = 无法控制预算 = 账单震惊。

In [ ]:
"""
成本追踪 —— 实时计算 LLM 调用成本
=====================================

定价数据（截至 2025 年中，以官方最新为准）：
价格单位：美元 / 1M tokens
"""
from dataclasses import dataclass, field
from datetime import datetime
from typing import List

# ============================================================
# 模型定价表（美元 / 1M tokens）
# ============================================================

# 注意：此为 2025 年中的大致定价，生产环境应从配置文件读取
MODEL_PRICING = {
    # OpenAI
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
    # Anthropic
    "claude-sonnet-4-6-20250514": {"input": 3.00, "output": 15.00},
    "claude-3-opus-20240229": {"input": 15.00, "output": 75.00},
    "claude-3-haiku-20240307": {"input": 0.25, "output": 1.25},
    "claude-3-5-sonnet-20241022": {"input": 3.00, "output": 15.00},
    # Embedding 模型
    "text-embedding-3-small": {"input": 0.02, "output": 0.0},
    "text-embedding-3-large": {"input": 0.13, "output": 0.0},
}


@dataclass
class CostEntry:
    """
    单次 API 调用的成本记录。
    
    每次 LLM 调用后创建一个 CostEntry 来追踪花费。
    """
    timestamp: datetime = field(default_factory=datetime.now)
    model: str = ""
    provider: str = ""              # openai / anthropic
    input_tokens: int = 0
    output_tokens: int = 0
    estimated_cost: float = 0.0     # 美元
    purpose: str = ""               # 调用目的（用于分类统计）


def calculate_cost(
    model: str,
    input_tokens: int,
    output_tokens: int = 0,
) -> float:
    """
    计算单次 API 调用的成本。
    
    Args:
        model: 模型名称
        input_tokens: 输入 token 数
        output_tokens: 输出 token 数（embedding 为 0）
    
    Returns:
        成本（美元）
    
    Raises:
        ValueError: 未知模型的成本为 0 时发出警告
    """
    pricing = MODEL_PRICING.get(model)
    if pricing is None:
        print(f"  ⚠️  未知模型 '{model}' 的定价，成本计为 $0")
        return 0.0
    
    input_cost = (input_tokens / 1_000_000) * pricing["input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    
    return input_cost + output_cost


# 演示：计算成本
print("=" * 60)
print("成本计算示例")
print("=" * 60)
print()

scenarios = [
    ("小幅对话", "gpt-4o-mini", 200, 100),
    ("小幅对话", "gpt-4o", 200, 100),
    ("中等文档分析", "gpt-4o", 2000, 500),
    ("长文档分析", "claude-sonnet-4-6-20250514", 10000, 2000),
    ("大规模 Embedding", "text-embedding-3-small", 100_000, 0),
]

print(f"{'场景':<16} {'模型':<28} {'输入':<10} {'输出':<10} {'成本(USD)':<12}")
print("-" * 76)

for purpose, model, inp, out in scenarios:
    cost = calculate_cost(model, inp, out)
    print(f"{purpose:<16} {model:<28} {inp:<10,} {out:<10,} ${cost:<12.6f}")

print()
print(f"💡 GPT-4o-mini 比 GPT-4o 便宜约 ~16x (输入), ~16x (输出)")

In [ ]:
# ============================================================
# 会话级成本追踪器
# ============================================================

@dataclass
class CostTracker:
    """
    会话级成本追踪器。
    
    功能：
    - 记录每次 API 调用
    - 按目的分类统计
    - 实时显示累计成本
    - 预算告警
    
    用法：
    ```python
    tracker = CostTracker(budget_limit=1.00)
    
    # 每次 API 调用后
    tracker.record(
        model="gpt-4o",
        provider="openai",
        input_tokens=500,
        output_tokens=200,
        purpose="generation",
    )
    
    tracker.report()  # 打印报告
    ```
    """
    budget_limit: float = 5.0  # 美元
    entries: List[CostEntry] = field(default_factory=list)
    warn_threshold: float = 0.8  # 80% 预算时警告
    
    @property
    def total_cost(self) -> float:
        """累计成本"""
        return sum(e.estimated_cost for e in self.entries)
    
    @property
    def total_input_tokens(self) -> int:
        """累计输入 tokens"""
        return sum(e.input_tokens for e in self.entries)
    
    @property
    def total_output_tokens(self) -> int:
        """累计输出 tokens"""
        return sum(e.output_tokens for e in self.entries)
    
    @property
    def budget_remaining(self) -> float:
        """剩余预算"""
        return self.budget_limit - self.total_cost
    
    @property
    def is_over_budget(self) -> bool:
        """是否超出预算"""
        return self.total_cost >= self.budget_limit
    
    def record(
        self,
        model: str,
        provider: str,
        input_tokens: int,
        output_tokens: int = 0,
        purpose: str = "general",
    ) -> CostEntry:
        """
        记录一次 API 调用的成本。
        
        Returns:
            创建的 CostEntry
        """
        cost = calculate_cost(model, input_tokens, output_tokens)
        check_cost = min(cost, self.budget_remaining) if self.budget_remaining > 0 else 0
        
        entry = CostEntry(
            model=model,
            provider=provider,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            estimated_cost=cost,
            purpose=purpose,
        )
        self.entries.append(entry)
        
        # 预算警告
        if self.total_cost >= self.budget_limit * self.warn_threshold:
            if not self.is_over_budget:
                pct = (self.total_cost / self.budget_limit) * 100
                print(f"  ⚠️  预算使用: ${self.total_cost:.4f} / ${self.budget_limit:.2f} ({pct:.0f}%)")
        
        if self.is_over_budget:
            print(f"  🚨 已超出预算! ${self.total_cost:.4f} > ${self.budget_limit:.2f}")
        
        return entry
    
    def cost_by_provider(self) -> Dict[str, float]:
        """按 Provider 统计成本"""
        result: Dict[str, float] = {}
        for entry in self.entries:
            result[entry.provider] = result.get(entry.provider, 0.0) + entry.estimated_cost
        return result
    
    def cost_by_purpose(self) -> Dict[str, float]:
        """按目的统计成本"""
        result: Dict[str, float] = {}
        for entry in self.entries:
            result[entry.purpose] = result.get(entry.purpose, 0.0) + entry.estimated_cost
        return result
    
    def cost_by_model(self) -> Dict[str, float]:
        """按模型统计成本"""
        result: Dict[str, float] = {}
        for entry in self.entries:
            result[entry.model] = result.get(entry.model, 0.0) + entry.estimated_cost
        return result
    
    def report(self) -> str:
        """生成完整成本报告"""
        lines = []
        lines.append("=" * 60)
        lines.append("💰 会话成本报告")
        lines.append("=" * 60)
        lines.append(f"  总调用次数: {len(self.entries)}")
        lines.append(f"  总输入 tokens: {self.total_input_tokens:,}")
        lines.append(f"  总输出 tokens: {self.total_output_tokens:,}")
        lines.append(f"  总成本: ${self.total_cost:.6f}")
        lines.append(f"  预算: ${self.budget_limit:.2f} | 剩余: ${self.budget_remaining:.4f}")
        lines.append(f"  使用率: {(self.total_cost/self.budget_limit)*100:.1f}%")
        lines.append("")
        lines.append("  按 Provider:")
        for provider, cost in self.cost_by_provider().items():
            lines.append(f"    {provider}: ${cost:.6f}")
        lines.append("  按目的:")
        for purpose, cost in self.cost_by_purpose().items():
            lines.append(f"    {purpose}: ${cost:.6f}")
        lines.append("  按模型:")
        for model, cost in self.cost_by_model().items():
            lines.append(f"    {model}: ${cost:.6f}")
        lines.append("=" * 60)
        return "\n".join(lines)
    
    def reset(self) -> None:
        """重置追踪器（开始新会话）"""
        self.entries.clear()


# 演示
print("=" * 60)
print("会话成本追踪演示")
print("=" * 60)
print()

# 创建追踪器，预算 $0.50
tracker = CostTracker(budget_limit=0.50)

# 模拟一系列 API 调用
tracker.record(
    model="gpt-4o-mini", provider="openai",
    input_tokens=500, output_tokens=200,
    purpose="classification",
)
tracker.record(
    model="gpt-4o", provider="openai",
    input_tokens=3000, output_tokens=800,
    purpose="generation",
)
tracker.record(
    model="claude-sonnet-4-6-20250514", provider="anthropic",
    input_tokens=2000, output_tokens=500,
    purpose="analysis",
)
tracker.record(
    model="gpt-4o-mini", provider="openai",
    input_tokens=300, output_tokens=100,
    purpose="classification",
)

# 打印报告
print(tracker.report())

print()
print("💡 CostTracker 可集成到 FastAPI/Flask 的 middleware 中实现全站成本追踪")

### ✏️ 练习 7: 成本追踪

1. 扩展 `CostTracker` 添加 `to_csv(path)` 方法导出成本记录。
2. 实现 `estimate_cost_before_call(prompt, model, max_output_tokens)` 在调用前估算成本。
3. 添加日/周/月统计功能（按时间聚合）。
4. 实现成本优化建议：自动选择更便宜的模型当检测到简单任务时。

---
## 📋 学习检查清单

完成本 Notebook 后，你应该能够：

- [ ] 独立编写完整的 OpenAI Chat Completion 调用代码
- [ ] 实现流式输出并在 UI 中实时显示
- [ ] 使用 tiktoken 精确计算 Token 并在请求前验证预算
- [ ] 区分 HTTP 401/429/500/503 并正确分类可重试/不可重试
- [ ] 实现指数退避 + Full Jitter 重试策略（不依赖 tenacity）
- [ ] 调用 Anthropic Messages API 并理解与 OpenAI 的差异
- [ ] 并发调用多个 Provider 并对比响应质量
- [ ] 实现会话级成本追踪和预算告警

**下一步：** 03-vector-embeddings.ipynb —— 使用这些 API 技能构建向量检索系统